# KalKalori — BareTubeHeatExchanger Test (with Δp)

Minimal end-to-end validation notebook.

In [ ]:
import sys
import os

# Ensure repository root is on sys.path so `core` is importable when running this notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Imports — KalKalori BareTubeHeatExchanger test

from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle

from core.heat_transfer.internal_flow import (
    FluidProps as TubeFluidProps,
)

from core.heat_transfer.outside_flow import (
    FluidProps as OutsideFluidProps,
)

from core.heat_transfer.streams import (
    SensibleHeatStream,
)

from core.models.bare_tube import (
    BareTubeHeatExchanger,
)

In [2]:
# Geometry

tube = BareTube(
    D_i=0.015,   # m
    D_o=0.018,   # m
    length_total=1.34,      # m
    length_effective=1.33,  # m
    wall_k=16.0,            # W/(m*K), steel
)

bundle = TubeBundle(
    tube=tube,
    n_rows=4,
    n_tubes_per_row=20,
    pitch_transverse=0.025,
    pitch_longitudinal=0.025,
    layout="inline",    # "inline" / "staggered"
    n_passes_tube=2,
    flow_arrangement="counterflow", # "crossflow" / "counterflow" / "cocurrentflow"
)

In [3]:
# Tube-side fluid (water)

tube_props = TubeFluidProps(
    rho=973.0,        # kg/m3
    mu=0.00036,       # Pa*s
    k=0.67,           # W/(m*K)
    cp=4196.0        # J/(kg*K)
)

m_dot_water = 3.0   # kg/s


In [4]:
# Outside forced flow (air)

outside_props = OutsideFluidProps(
    rho=1.16,        # kg/m3
    mu=0.00002,       # Pa*s
    k=0.03,         # W/(m*K)
    cp=1026.4        # J/(kg*K)
)

m_dot_air = 5.5  # kg/s


In [5]:
hot_stream = SensibleHeatStream(
    C=m_dot_water * tube_props.cp,
    T_in=273.15 + 80       # K
)

cold_stream = SensibleHeatStream(
    C=m_dot_air * outside_props.cp,
    T_in=273.15 + 20       # K
)


In [6]:
# Heat exchanger model

hx = BareTubeHeatExchanger(bundle=bundle)  # wall_k now supplied by the tube geometry

In [7]:
import inspect
from core.models.bare_tube import BareTubeHeatExchanger
import core.models.bare_tube as bt

print("bare_tube.py loaded from:", bt.__file__)
print("solve signature:", inspect.signature(BareTubeHeatExchanger.solve))


bare_tube.py loaded from: C:\Users\pawel\GitHub\kalkalori\core\models\bare_tube.py
solve signature: (self, hot_stream: 'EnergyStream', cold_stream: 'EnergyStream', *, m_dot_tube_side: 'float', tube_side_props: 'InternalFlowFluidProps', tube_side_provider: 'PropertyProvider | None' = None, tube_side_temperature_in: 'float | None' = None, tube_side_temperature_out: 'float | None' = None, tube_side_pressure: 'float | None' = None, m_dot_outside: 'float | None' = None, outside_props: 'OutsideFlowFluidProps | None' = None, outside_provider: 'PropertyProvider | None' = None, outside_temperature_in: 'float | None' = None, outside_temperature_out: 'float | None' = None, outside_pressure: 'float | None' = None, K_inlet: 'float' = 0.5, K_outlet: 'float' = 1.0, K_turn: 'float' = 1.5, alfa_o: 'float | None' = None, flow_arrangement: 'str | None' = None, euler_provider: 'str | EulerProvider' = 'zukauskas') -> 'HXResult'


In [8]:
result = hx.solve(
    hot_stream=hot_stream,
    cold_stream=cold_stream,
    m_dot_tube_side=m_dot_water,
    tube_side_props=tube_props,
    m_dot_outside=m_dot_air,
    outside_props=outside_props,
    euler_provider="zukauskas", # "zukauskas", "kern", "esdu", gaddis_gnielinski
)

In [9]:
# Print full results object

print("=== HEAT EXCHANGER RESULTS ===\n")

print("Input streams:")
print(f"  Hot side : T_in = {hot_stream.inlet_temperature() - 273.15:.2f} °C,  C = {hot_stream.capacity_rate():.1f} W/K")
print(f"  Cold side: T_in = {cold_stream.inlet_temperature() - 273.15:.2f} °C,  C = {cold_stream.capacity_rate():.1f} W/K\n")

print("Geometry:")
print(f"  Tube passes       : {hx.bundle.n_passes_tube}")
print(f"  Flow arrangement  : {hx.bundle.flow_arrangement}")
print(f"  Tubes total       : {hx.bundle.n_tubes_total}")
print(f"  Rows              : {hx.bundle.n_rows}")
print(f"  Length effective  : {hx.bundle.tube.length_effective:.2f} m")
print(f"  Length total      : {hx.bundle.tube.length_total:.2f} m\n")

print("Heat transfer areas:")
print(f"  A_i       = {result.A_i:.3f} m^2")
print(f"  A_o       = {result.A_o:.3f} m^2")
print(f"  A_frontal = {result.A_frontal:.3f} m^2\n")

print("Thermal performance:")
print(f"  UA   = {result.UA:.1f} W/K")
print(f"  eps  = {result.eps:.3f}")
print(f"  Q    = {result.Q:.1f} W")
print(f"  T_hot_out  = {result.T_hot_out - 273.15:.2f} °C")
print(f"  T_cold_out = {result.T_cold_out - 273.15:.2f} °C\n")

print("Tube-side (internal):")
print(f"  v    = {result.tube_side_thermal.v:.2f} m/s")
print(f"  Re   = {result.tube_side_thermal.Re:.0f}")
print(f"  alfa = {result.tube_side_thermal.alfa:.1f} W/m^2/K")
tube_hyd = result.tube_side_hydraulic.tube_bundle
print(f"  Tube path type      = {tube_hyd.tube_path_type.value}")
print(f"  Tube-side straight-tube friction pressure drop      = {tube_hyd.dp_straight_tube_friction:.1f} Pa")
print(f"  Tube-side straight-tube acceleration pressure change = {tube_hyd.dp_straight_tube_acceleration:.1f} Pa")
print(f"  Tube-side straight-tube total pressure change        = {tube_hyd.dp_straight_tubes:.1f} Pa")
print(f"  Tube entrance pressure drop ({tube_hyd.entrance_count} entrance(s)) = {tube_hyd.dp_tube_entrances:.1f} Pa")
print(f"  Tube exit pressure drop     ({tube_hyd.exit_count} exit(s))      = {tube_hyd.dp_tube_exits:.1f} Pa")
print(f"  Tube-bundle pressure drop (straight + entrances + exits) = {tube_hyd.dp_tube_bundle:.1f} Pa")
print(f"  midpoint method       = {tube_hyd.midpoint_method}")
print(f"  pass-boundary method  = {tube_hyd.pass_boundary_method}")
print(f"  flow area/pass        = {tube_hyd.flow_area_per_pass:.6g} m^2")
print(f"  mass flux             = {tube_hyd.mass_flux:.6g} kg/m^2/s")
print(f"  hydraulic diameter    = {tube_hyd.hydraulic_diameter:.6g} m")
print(f"  hydraulic length      = {tube_hyd.hydraulic_length_total:.6g} m")
for label, point in (("inlet", tube_hyd.inlet), ("midpoint", tube_hyd.midpoint), ("outlet", tube_hyd.outlet)):
    print(f"  {label:8s}: T={point.temperature:.2f} K, rho={point.props.rho:.6g} kg/m^3, mu={point.props.mu:.6g} Pa*s, v={point.velocity:.6g} m/s, Re={point.reynolds:.6g}, f_D={point.friction_factor:.6g}, q={point.dynamic_pressure:.6g} Pa")
print("  Pass-boundary states:")
for state in tube_hyd.pass_boundary_states:
    print(f"    boundary {state.boundary_index}: T={state.temperature:.2f} K, rho={state.props.rho:.6g} kg/m^3, v={state.velocity:.6g} m/s, q={state.dynamic_pressure:.6g} Pa, Re={state.reynolds:.6g}")
print("  Tube-sheet entrances:")
for entry in tube_hyd.entrance_results:
    print(f"    {entry.component_id}: boundary={entry.boundary_index}, K={entry.loss_coefficient:.3g}, v_ref={entry.reference_velocity:.6g} m/s, q_ref={entry.reference_dynamic_pressure:.6g} Pa, dp={entry.pressure_drop:.3g} Pa, method={entry.method}")
print("  Tube-sheet exits:")
for exit_ in tube_hyd.exit_results:
    print(f"    {exit_.component_id}: boundary={exit_.boundary_index}, K={exit_.loss_coefficient:.3g}, v_ref={exit_.reference_velocity:.6g} m/s, q_ref={exit_.reference_dynamic_pressure:.6g} Pa, dp={exit_.pressure_drop:.3g} Pa, method={exit_.method}")
print("Tube-bundle pressure drop includes straight-tube friction, tube-side acceleration, and tube-sheet entrance and exit losses. Return chambers, U-bends, nozzles, chambers, transitions, and other additional local losses are not included in the standard calculation.\n")

# Pressure-drop flow-path architecture (v0.5.6): stage-by-stage/grouped
# aggregation, read entirely from the production result object -- no
# equation is reconstructed here. dp_core/dp_local/dp_total are
# irreversible losses; signed dynamic and static changes are separate.
# The standard solver defines no local path, so dp_local is exactly zero.
tube_dp = result.tube_side_pressure_drop
print("Tube-side pressure-drop flow path:")
print(f"  Tube-side core irreversible loss             (dp_core)  = {tube_dp.dp_core:.1f} Pa")
print(f"  Tube-side local irreversible loss            (dp_local) = {tube_dp.dp_local:.1f} Pa")
print(f"  Tube-side total irreversible loss            (dp_total) = {tube_dp.dp_total:.1f} Pa")
print(f"  Tube-side total dynamic-pressure change                  = {tube_dp.delta_dynamic_pressure_total:.1f} Pa")
print(f"  Tube-side signed static-pressure difference              = {tube_dp.dp_static_total:.1f} Pa")
for group in tube_dp.flow_path.groups:
    statuses = ", ".join(f"{stage.stage_id}[{stage.status.value}]" for stage in group.stages) or "(no stages)"
    print(f"    group '{group.group_id}': {statuses}")
print()

print("Outside-side:")
print(f"  v_face (thermal) = {result.outside_side_thermal.v:.2f} m/s")
print(f"  Re (thermal)     = {result.outside_side_thermal.Re:.0f}")
print(f"  alfa             = {result.outside_side_thermal.alfa:.1f} W/m^2/K")
outside_hyd = result.outside_side_hydraulic.tube_bank
print(f"  Outside tube-bank pressure drop = {outside_hyd.dp_total:.1f} Pa")
print(f"    drag                          = {outside_hyd.dp_drag:.1f} Pa")
print(f"    acceleration                  = {outside_hyd.dp_acceleration:.1f} Pa")
print(f"  face area / G    = {outside_hyd.face_area:.6g} m^2 / {outside_hyd.face_mass_flux:.6g} kg/(m^2 s)")
print(f"  ref area / G     = {outside_hyd.reference_area:.6g} m^2 / {outside_hyd.reference_mass_flux:.6g} kg/(m^2 s)")
print(f"  ref velocity     = {outside_hyd.reference_velocity_definition}")
print(f"  Euler basis/rows = {outside_hyd.euler_basis} / {outside_hyd.n_rows_effective:.6g}")
print(f"  midpoint method  = {outside_hyd.midpoint_method}")
for label, point in (("inlet", outside_hyd.inlet), ("midpoint", outside_hyd.midpoint), ("outlet", outside_hyd.outlet)):
    print(f"  {label:8s}: T={point.temperature:.2f} K, rho={point.props.rho:.6g} kg/m^3, mu={point.props.mu:.6g} Pa*s, V_face={point.face_velocity:.6g} m/s, V_ref={point.reference_velocity:.6g} m/s, Re={point.reynolds:.6g}, Eu={point.euler_number:.6g}, q_ref={point.dynamic_pressure_reference:.6g} Pa")
print("Outside tube-bank pressure drop covers irreversible crossflow drag through the bare-tube bank and the signed inlet-to-outlet acceleration pressure change. Duct, plenum, casing-transition, screen, louver, and other external local losses are not included.\n")

outside_dp = result.outside_side_pressure_drop
print("Outside pressure-drop flow path:")
print(f"  Outside core irreversible loss             (dp_core)  = {outside_dp.dp_core:.1f} Pa")
print(f"  Outside local irreversible loss            (dp_local) = {outside_dp.dp_local:.1f} Pa")
print(f"  Outside total irreversible loss            (dp_total) = {outside_dp.dp_total:.1f} Pa")
print(f"  Outside total dynamic-pressure change                  = {outside_dp.delta_dynamic_pressure_total:.1f} Pa")
print(f"  Outside signed static-pressure difference              = {outside_dp.dp_static_total:.1f} Pa")
for group in outside_dp.flow_path.groups:
    statuses = ", ".join(f"{stage.stage_id}[{stage.status.value}]" for stage in group.stages) or "(no stages)"
    print(f"    group '{group.group_id}': {statuses}")
print("\n")
# Display warnings if any
if result.warnings:
    print("⚠️  WARNINGS:")
    for i, warning in enumerate(result.warnings, 1):
        print(f"  {i}. {warning}")
else:
    print("✓ No warnings - all parameters within expected ranges")


=== HEAT EXCHANGER RESULTS ===

Input streams:
  Hot side : T_in = 80.00 °C,  C = 12588.0 W/K
  Cold side: T_in = 20.00 °C,  C = 5645.2 W/K

Geometry:
  Tube passes       : 2
  Flow arrangement  : counterflow
  Tubes total       : 80
  Rows              : 4
  Length effective  : 1.33 m
  Length total      : 1.34 m

Heat transfer areas:
  A_i       = 5.014 m^2
  A_o       = 6.017 m^2
  A_frontal = 0.665 m^2

Thermal performance:
  UA   = 1180.8 W/K
  eps  = 0.181
  Q    = 61470.1 W
  T_hot_out  = 75.12 °C
  T_cold_out = 30.89 °C

Tube-side (internal):
  v    = 0.44 m/s
  Re   = 17684
  alfa = 3703.6 W/m^2/K
  Tube path type      = straight
  Tube-side straight-tube friction pressure drop      = 446.4 Pa
  Tube-side straight-tube acceleration pressure change = 0.0 Pa
  Tube-side straight-tube total pressure change        = 446.4 Pa
  Tube entrance pressure drop (2 entrance(s)) = 92.6 Pa
  Tube exit pressure drop     (2 exit(s))      = 185.1 Pa
  Tube-bundle pressure drop (straight + entr

## Inlet, midpoint, and outlet fluid properties

Point states below come directly from the solver's existing hydraulic results. The wet midpoint uses arithmetic mean temperature and water ratio; no transport-property provider is called for presentation.

In [10]:
import math
import pandas as pd


def endpoint_property_table(solver_result, side):
    """Read solver-owned hydraulic point states without provider calls."""
    states = (
        ("inlet", getattr(solver_result, f"{side}_properties_inlet")),
        ("midpoint", getattr(solver_result, f"{side}_properties_midpoint")),
        ("outlet", getattr(solver_result, f"{side}_properties_outlet")),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "T [°C]": state.T - 273.15,
                "p [Pa]": state.p,
                "rho [kg/m³]": state.rho,
                "cp [J/(kg·K)]": state.cp,
                "mu [Pa·s]": state.mu,
                "k [W/(m·K)]": state.k,
                "Pr [-]": state.Pr,
            }
            for name, state in states
            if state is not None
        ]
    ).set_index("state")


def representative_0d_property_table(solver_result):
    """Keep representative thermal properties separate from point states."""
    if hasattr(solver_result, "inside_props_mean"):
        pairs = (
            ("inside", solver_result.T_mean_inside, solver_result.inside_props_mean),
            ("outside", solver_result.T_mean_outside, solver_result.outside_props_mean),
        )
    elif getattr(solver_result, "thermal_state", None) is not None:
        thermal = solver_result.thermal_state
        pairs = (
            ("inside", thermal.inside_bulk_temperature, thermal.inside_bulk_props),
            ("outside", thermal.outside_bulk_temperature, thermal.outside_bulk_props),
        )
    else:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "side": side,
                "T representative [°C]": temperature - 273.15,
                "rho [kg/m³]": props.rho,
                "cp [J/(kg·K)]": props.cp,
                "mu [Pa·s]": props.mu,
                "k [W/(m·K)]": props.k,
                "Pr [-]": props.mu * props.cp / props.k,
            }
            for side, temperature, props in pairs
        ]
    ).set_index("side")


def wet_gas_state_table(solver_result, outside_provider_for_result=None):
    """Combine hydraulic states with wet diagnostics already returned by the solver."""
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable or pc.W_in is None:
        return pd.DataFrame()

    W_mid = 0.5 * (pc.W_in + pc.W_out)
    dew_mid = math.nan
    if outside_provider_for_result is not None:
        from core.phase_change.capability import detect_phase_change_capability
        from core.phase_change.integration import _dew_point_at_ratio

        capability = detect_phase_change_capability(outside_provider_for_result)
        midpoint_state = solver_result.outside_properties_midpoint
        dew_mid_value = _dew_point_at_ratio(
            capability, W_mid, p=midpoint_state.p
        )
        dew_mid = math.nan if dew_mid_value is None else dew_mid_value

    dry_flow = pc.m_dot_dry_carrier
    vapor_mid = (
        math.nan
        if dry_flow is None
        else dry_flow * W_mid
    )
    gas_mid = (
        math.nan
        if dry_flow is None
        else dry_flow + vapor_mid
    )
    values = (
        ("inlet", pc.W_in, pc.dew_point_in, pc.m_dot_gas_in, pc.m_dot_water_vapor_in),
        ("midpoint", W_mid, dew_mid, gas_mid, vapor_mid),
        ("outlet", pc.W_out, pc.dew_point_out, pc.m_dot_gas_out, pc.m_dot_water_vapor_out),
    )
    return pd.DataFrame(
        [
            {
                "state": name,
                "W [kg/kg dry]": W,
                "dew point [°C]": (
                    math.nan if dew_point is None else dew_point - 273.15
                ),
                "m_dot gas [kg/s]": gas_flow,
                "m_dot water vapor [kg/s]": vapor_flow,
            }
            for name, W, dew_point, gas_flow, vapor_flow in values
        ]
    ).set_index("state")


def condensation_summary_table(solver_result):
    pc = getattr(solver_result, "outside_phase_change", None)
    if pc is None or not pc.capable:
        return pd.DataFrame()
    return pd.DataFrame(
        [
            {
                "m_dot condensate [kg/s]": pc.m_dot_condensate,
                "Q_sensible [W]": pc.Q_sensible,
                "Q_latent [W]": pc.Q_latent,
                "wet_surface_fraction [-]": pc.wet_surface_fraction,
                "wall Tmin [°C]": (
                    math.nan if pc.wall_temperature_min is None
                    else pc.wall_temperature_min - 273.15
                ),
                "wall Tmean [°C]": (
                    math.nan if pc.wall_temperature_mean is None
                    else pc.wall_temperature_mean - 273.15
                ),
                "wall Tmax [°C]": (
                    math.nan if pc.wall_temperature_max is None
                    else pc.wall_temperature_max - 273.15
                ),
            }
        ],
        index=["outside"],
    )

In [11]:
endpoint_results = [('HXResult', result)]
outside_provider_for_endpoint_table = None

for result_label, endpoint_result in endpoint_results:
    print(result_label)
    print("Inside")
    display(endpoint_property_table(endpoint_result, "inside"))
    print("Outside")
    display(endpoint_property_table(endpoint_result, "outside"))

    wet_table = wet_gas_state_table(
        endpoint_result, outside_provider_for_endpoint_table
    )
    if not wet_table.empty:
        print("Outside wet-gas mass and dew-point diagnostics")
        display(wet_table)
        display(condensation_summary_table(endpoint_result))

HXResult
Inside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,26.85,101325.0,973.0,4196.0,0.00036,0.67,2.254567
midpoint,26.85,101325.0,973.0,4196.0,0.00036,0.67,2.254567
outlet,26.85,101325.0,973.0,4196.0,0.00036,0.67,2.254567


Outside


,T [°C],p [Pa],rho [kg/m³],cp [J/(kg·K)],mu [Pa·s],k [W/(m·K)],Pr [-]
state,,,,,,,
inlet,26.85,101325.0,1.16,1026.4,0.00002,0.03,0.684267
midpoint,26.85,101325.0,1.16,1026.4,0.00002,0.03,0.684267
outlet,26.85,101325.0,1.16,1026.4,0.00002,0.03,0.684267


## Representative 0D properties used by the solver

These lumped thermal-model properties are retained separately; they are not substitutes for inlet or outlet states.

In [12]:
for result_label, endpoint_result in endpoint_results:
    representative_table = representative_0d_property_table(endpoint_result)
    if not representative_table.empty:
        print(result_label)
        display(representative_table)